# eFootball 6 类 YOLO · 云端一键训练

把仓库里的 `training/dataset.zip` 上传到本 Notebook，然后「运行所有单元格」即可。
全程会自动：解压 → 校验数据 → 训练 → 导出 ONNX → 校验结构 → 在误检帧上量 `main_menu` 分数 → 下载候选模型。

**验收线**（训练前基线，用现网模型实测）：周活动卡片页的 `main_menu` 分数 0.775 / 0.867 / 0.892 / 0.920，
比赛选择页 3 帧为 0.000。训练后周活动页应降到共享分类阈值 **0.45 以下**，同时真主菜单仍检得出。

**一致性**：参数与本地 `training/train_yolo.py` 对齐（epochs=100 / imgsz=640 / batch=16 / seed=0 / dynamic=False）。
若本地也要训一遍做对比，请用同一个 `dataset.zip`、同一个 `yolo26n.pt`、同一个 ultralytics 版本。

拿回 `yolo26n_candidate.onnx` 后放到仓库 `training/out/`，再执行本地 `verify_model.py` → 全量测试 → `train_yolo.py --promote`。

In [ ]:
# 1) 环境与 GPU 检查
import sys, platform, subprocess
print('Python', sys.version.split()[0], platform.platform())
try:
    import torch
    print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    else:
        print('!! 当前是 CPU 运行时：请到「运行时 -> 更改运行时类型」选 GPU（T4 即可）')
except ImportError:
    print('torch 未安装，下面会装')


In [ ]:
# 2) 依赖（固定打印版本，便于与本地对齐）
!pip install -q ultralytics onnx onnxruntime
import ultralytics, torch, onnx, onnxruntime as ort
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__)
print('onnx', onnx.__version__, '| onnxruntime', ort.__version__)


In [ ]:
# 3) 上传 dataset.zip（若已放在 /content 则跳过）；可选上传同一个 yolo26n.pt
from pathlib import Path
import zipfile, shutil

WORK = Path('/content/efootball_train')
WORK.mkdir(exist_ok=True)
zip_path = WORK / 'dataset.zip'
if not zip_path.exists():
    try:
        from google.colab import files
        up = files.upload()   # 选 training/dataset.zip
        name = next(iter(up))
        (WORK / 'dataset.zip').write_bytes(up[name])
    except ImportError:
        raise SystemExit('非 Colab 环境：请手动把 dataset.zip 放到 ' + str(zip_path))

data_dir = WORK / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir)
data_dir.mkdir()
with zipfile.ZipFile(zip_path) as z:
    z.extractall(data_dir)
print('解压完成，根目录：', sorted(p.name for p in data_dir.iterdir()))
assert (data_dir / 'data.yaml').is_file(), 'zip 结构不对：根目录应有 data.yaml'
print((data_dir / 'data.yaml').read_text(encoding='utf-8'))


In [ ]:
# 4) 数据集校验：类别顺序必须与仓库 classes.txt 一致（共享层硬编码了类名）
import re
EXPECTED = ['player_card','btn_back','btn_confirm','coin_bar','main_menu','item_row']
names = {}
for line in (data_dir / 'data.yaml').read_text(encoding='utf-8').splitlines():
    m = re.match(r'^\s*(\d+)\s*:\s*(\S+)\s*$', line)
    if m: names[int(m.group(1))] = m.group(2)
order = [names[i] for i in sorted(names)]
assert order == EXPECTED, f'类别顺序不对: {order}'
print('类别顺序 OK:', order)
for subset in ('train','val'):
    imgs = sorted((data_dir/'images'/subset).glob('*.*'))
    lbls = {p.stem for p in (data_dir/'labels'/subset).glob('*.txt')}
    missing = [p.stem for p in imgs if p.stem not in lbls]
    print(f'{subset}: {len(imgs)} 帧, 缺标签 {len(missing)}')
    assert not missing, missing[:5]
    print('  其中误检负样本:', sum(1 for p in imgs if p.stem.startswith('neg_')))


In [ ]:
# 5) 训练（参数与 training/train_yolo.py 完全一致）
from ultralytics import YOLO
WEIGHTS = 'yolo26n.pt'   # 若要与本地严格一致，请上传同一个 .pt 并改成它的路径
EPOCHS, IMGSZ, BATCH, SEED = 100, 640, 16, 0
model = YOLO(WEIGHTS)
results = model.train(
    data=str(data_dir / 'data.yaml'),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    seed=SEED, deterministic=False,
    project=str(WORK / 'runs'), name='efootball6', exist_ok=True,
)
print('训练完成；best.pt:', WORK/'runs'/'efootball6'/'weights'/'best.pt')


In [ ]:
# 6) 导出 ONNX（固定 imgsz / 静态形状，YoloModelSmokeTests 断言 640x640 + 6类）
exported = model.export(format='onnx', imgsz=IMGSZ, dynamic=False, simplify=False)
candidate = WORK / 'yolo26n_candidate.onnx'
shutil.copy2(exported, candidate)
print('候选模型:', candidate, f'({candidate.stat().st_size/1024/1024:.1f} MB)')

sess = ort.InferenceSession(str(candidate), providers=['CPUExecutionProvider'])
i, o = sess.get_inputs()[0], sess.get_outputs()[0]
print('输入', i.name, i.shape, '| 输出', o.name, o.shape)
assert len(i.shape) == 4 and i.shape[1] == 3, '输入应为 NCHW 3 通道'
assert i.shape[2] == IMGSZ and i.shape[3] == IMGSZ, '输入尺寸应为 640x640'
shp = list(o.shape)
assert len(shp) == 3, '输出应为 3 维'
ok = (shp[-1] == 6 or shp[1] == 6) or (shp[1] == 10 or shp[-1] == 10)
assert ok, f'输出形状与 6 类不匹配: {shp}'
print('结构校验通过（C# YoloDetector 兼容 [1,N,6] 与 [1,10,N]）')


In [ ]:
# 7) 云端直接验收：对 neg_* 误检帧跑一遍，输出 main_menu 最高分
import numpy as np
from PIL import Image
MENU_ID = EXPECTED.index('main_menu')  # 4
DETECT_TH, CLASSIFY_TH = 0.25, 0.45
w, h = int(i.shape[3]), int(i.shape[2])
shp = list(o.shape)
nms_free = (shp[-1] == 6 or shp[1] == 6) and not (shp[1] == 10 or shp[-1] == 10)

def letterbox(p):
    im = Image.open(p).convert('RGB')
    sw, sh = im.size
    s = min(w/sw, h/sh)
    nw, nh = round(sw*s), round(sh*s)
    canvas = Image.new('RGB', (w,h), (114,114,114))
    canvas.paste(im.resize((nw,nh), Image.Resampling.BILINEAR), ((w-nw)//2,(h-nh)//2))
    arr = np.asarray(canvas, dtype=np.float32)/255.0
    return arr.transpose(2,0,1)[None,...]

def menu_score(p):
    raw = sess.run(None, {i.name: letterbox(p)})[0][0]
    if nms_free:
        rows = raw.T if raw.shape[0] == 6 else raw
        hit = rows[(np.round(rows[:,5]).astype(int) == MENU_ID) & (rows[:,4] >= DETECT_TH)]
        return float(hit[:,4].max()) if len(hit) else 0.0
    rows = raw.T if raw.shape[0] == 10 else raw   # -> [N,10] = 4 box + 6 class
    col = 4 + MENU_ID
    hit = rows[rows[:,col] >= DETECT_TH]
    return float(hit[:,col].max()) if len(hit) else 0.0

frames = sorted([p for sub in ('train','val') for p in (data_dir/'images'/sub).glob('neg_*')])
print(f"{'帧':<40} {'main_menu':>10}  判定")
worst = 0.0
for p in frames:
    s = menu_score(p); worst = max(worst, s)
    print(f'{p.name:<40} {s:>10.3f}  ' + ('仍会误判' if s >= CLASSIFY_TH else 'OK'))
print(f'\n最差分数 {worst:.3f}（基线最高 0.920，目标 < {CLASSIFY_TH}）')
print('注意：这里只测负样本。真主菜单是否还能检出，请拿回本地跑全量 *RealFrameTests 确认。')


In [ ]:
# 8) 下载候选模型
try:
    from google.colab import files
    files.download(str(candidate))
except ImportError:
    print('非 Colab 环境，模型在:', candidate)


## 拿回本地后

1. 把下载的 `yolo26n_candidate.onnx` 放到仓库 `training/out/`。
2. `python training/verify_model.py --frames training/negatives/images --compare src/EfootballBot/models/yolo/yolo26n.onnx`
3. 全量 `dotnet test`（`YoloModelSmokeTests` 断言 640×640 + 6类；十几个 `*RealFrameTests` 直接加载这个 onnx）。
4. 若置信度分布变了，校准 `Yolo.ScoreThreshold`(0.25) 与 `MatchBrain.MinScore`(0.45f)。
5. 确认无误才 `python training/train_yolo.py --promote`（会先备份旧模型），然后发版。

**别跳过第 3 步**：负样本降下去只说明误报被压住，真主菜单/按钮类的召回要靠全量实机帧回归保证。